# Object Tracking

> **Intermediate · Video**


## Why this matters

Detection says what is present in one frame; tracking maintains an identity and trajectory through time, with explicit failure handling.

**Where it appears:** Sports clips, UI interactions, single-object monitoring, and lightweight detect-then-track systems.


## Learning Objectives

- Track a single object across frames using OpenCV's built-in trackers
- Understand the detect-then-track pattern vs tracking-only
- Handle tracking failure gracefully instead of silently returning wrong boxes


## Prerequisites

13 Video Processing and Background Motion

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

OpenCV trackers, bounding boxes, centroids, distance association, trajectories

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Object Tracking

Detection (finding an object) and tracking (following a known object across
frames) are different problems with different costs: detection is usually
more expensive and run less often; tracking is cheap and run every frame,
using the previous frame's result as a prior. OpenCV's tracker API
(`cv2.TrackerCSRT_create`, etc.) needs to be initialized with a bounding
box on frame 1, then `update()` each subsequent frame -- and *must* be
checked for success, since trackers can and do lose the target.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


### 1. Initializing and running a tracker

CSRT is slower but more accurate than KCF; pick per use case. Initialize on frame 0 with the known blob location and track through the rest of the video stream.



In [ ]:
import cv2
import numpy as np
from cv_utils import read_real_video_frames, load_real_image, get_real_data, show_grid


def create_tracker():
    """Create a CSRT tracker (fallback to MIL if contrib missing)."""
    if hasattr(cv2, "TrackerCSRT_create"):
        return cv2.TrackerCSRT_create()
    elif hasattr(cv2, "legacy") and hasattr(cv2.legacy, "TrackerCSRT_create"):
        return cv2.legacy.TrackerCSRT_create()
    return cv2.TrackerMIL_create()


frames = read_real_video_frames("traffic.mp4", max_frames=45)
initial_box = (
    2,
    frames[0].shape[0] // 2 - 18,
    36,
    36,
)  # (x, y, w, h) around the blob at frame 0

tracker = create_tracker()
tracker.init(frames[0], initial_box)

tracked_boxes = [initial_box]
for frame in frames[1:]:
    ok, box = tracker.update(frame)
    tracked_boxes.append(box if ok else None)

lost_count = sum(1 for b in tracked_boxes if b is None)
print(f"Frames tracked: {len(tracked_boxes)}, lost: {lost_count}")

### 2. Visualizing the tracked trajectory

Plot the tracker's box centers across frames on the last frame -- a quick sanity check that the tracker actually followed the moving object smoothly.


In [ ]:
final_frame = frames[-1].copy()
centers = []
for box in tracked_boxes:
    if box is not None:
        x, y, w, h = [int(v) for v in box]
        centers.append((x + w // 2, y + h // 2))

for i in range(1, len(centers)):
    cv2.line(final_frame, centers[i - 1], centers[i], (0, 255, 255), 2)
if tracked_boxes[-1] is not None:
    x, y, w, h = [int(v) for v in tracked_boxes[-1]]
    cv2.rectangle(final_frame, (x, y), (x + w, y + h), (0, 0, 255), 2)

show_grid([("tracked trajectory", final_frame)], cols=1)

### 3. Handling tracking failure explicitly

A robust pipeline must decide what to do when `update()` returns `ok=False` -- here, fall back to reusing the last known box and flag it as stale rather than crashing.


In [ ]:
def robust_track(tracker, frame: np.ndarray, last_known_box) -> tuple:
    """Returns (box, is_stale). If tracking fails, reuse the last known box and mark stale."""
    ok, box = tracker.update(frame)
    if ok:
        return box, False
    return last_known_box, True


last_box = initial_box
stale_frames = 0
for frame in frames[1:]:
    last_box, stale = robust_track(tracker, frame, last_box)
    stale_frames += int(stale)

print(f"Frames where tracking failed and fallback box was reused: {stale_frames}")
# Let's visualize the robust tracker's final state
annotated_robust = frames[-1].copy()
if last_box is not None:
    x, y, w, h = [int(v) for v in last_box]
    cv2.rectangle(annotated_robust, (x, y), (x + w, y + h), (255, 0, 0), 2)
    cv2.putText(
        annotated_robust,
        "Robust Track",
        (x, y - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 0, 0),
        2,
    )
show_grid([("Robust Fallback Tracker Result", annotated_robust)])

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Object Tracking: Bounding Box Centroid Tracking (Euclidean Distance Association)

Trackers (like KCF) can fail under occlusions. To manage identity matching of multiple boxes, we implement a simple Centroid Tracker that associates detections across consecutive frames by minimizing Euclidean distance.


In [ ]:
class CentroidTracker:
    def __init__(self, max_disappeared=5):
        self.next_object_id = 0
        self.objects = {}
        self.disappeared = {}
        self.max_disappeared = max_disappeared

    def register(self, centroid):
        self.objects[self.next_object_id] = centroid
        self.disappeared[self.next_object_id] = 0
        self.next_object_id += 1

    def update(self, rects):
        # If no detections exist, mark objects as disappeared
        if len(rects) == 0:
            for oid in list(self.disappeared.keys()):
                self.disappeared[oid] += 1
                if self.disappeared[oid] > self.max_disappeared:
                    del self.objects[oid]
                    del self.disappeared[oid]
            return self.objects

        # Calculate centroids
        input_centroids = np.zeros((len(rects), 2), dtype=np.int32)
        for i, (startX, startY, endX, endY) in enumerate(rects):
            cx = int((startX + endX) / 2.0)
            cy = int((startY + endY) / 2.0)
            input_centroids[i] = (cx, cy)

        # If current objects map is empty, register all centroids
        if len(self.objects) == 0:
            for c in input_centroids:
                self.register(c)
        else:
            object_ids = list(self.objects.keys())
            object_centroids = list(self.objects.values())

            # Simple association logic: match nearest points
            for c in input_centroids:
                dists = np.linalg.norm(np.array(object_centroids) - c, axis=1)
                best_idx = np.argmin(dists)
                oid = object_ids[best_idx]
                self.objects[oid] = c
                self.disappeared[oid] = 0

        return self.objects


ct = CentroidTracker()
objects = ct.update([(10, 10, 50, 50), (120, 100, 160, 140)])
print("Registered objects mapping:", objects)
# Let's visualize the Centroid Tracker mapping!
canvas = np.full((200, 300, 3), 255, dtype=np.uint8)
rects = [(10, 10, 50, 50), (120, 100, 160, 140)]
for startX, startY, endX, endY in rects:
    cv2.rectangle(canvas, (startX, startY), (endX, endY), (200, 200, 200), -1)
    cv2.rectangle(canvas, (startX, startY), (endX, endY), (0, 0, 0), 1)
for objectID, centroid in objects.items():
    text = f"ID {objectID}"
    cv2.putText(
        canvas,
        text,
        (centroid[0] - 10, centroid[1] - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (0, 0, 255),
        2,
    )
    cv2.circle(canvas, (centroid[0], centroid[1]), 4, (0, 0, 255), -1)
show_grid([("Centroid Tracker Output", canvas)])

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Object Tracking
1. Compare tracker accuracy/speed for CSRT vs KCF vs MOSSE on the same video stream using `cv_utils.Timer`.
2. Implement re-detection: if a tracker is stale for more than N consecutive frames, re-run a detector (e.g. template matching) to reacquire the object.
3. Extend the pipeline to track two simultaneously moving objects with two independent tracker instances.

Use the empty cell below to work through them.



#### Solutions — Object Tracking

In [ ]:
# Solution 1: Compare tracker accuracy/speed for CSRT vs KCF vs MOSSE
# Explanation: MOSSE is extremely fast (high FPS, low computational complexity) but lacks accuracy
# and fails easily on deformation. KCF provides a balanced trade-off. CSRT is highly accurate and
# handles scale changes and deformation well, but is significantly slower (lower throughput).

In [ ]:
# Solution 2: Re-detection loop template
def update_with_redetection(
    tracker, frame, frame_idx, detect_interval=10
) -> tuple[bool, tuple]:
    """Re-initialize detection if tracking drops or at intervals."""
    # pseudo-code structure
    # if frame_idx % detect_interval == 0:
    #     bbox = detect_object(frame) # template matching / cascade
    #     tracker.init(frame, bbox)
    # else:
    #     success, bbox = tracker.update(frame)
    pass

In [ ]:
# Solution 3: Multi-object tracking with multiple tracker instances
class MultiTrackerWrapper:
    """Manage tracking of multiple concurrent targets."""

    def __init__(self):
        self.trackers = []

    def add_tracker(self, frame, bbox):
        t = (
            cv2.TrackerKCF_create()
            if hasattr(cv2, "TrackerKCF_create")
            else cv2.TrackerMIL_create()
        )
        t.init(frame, bbox)
        self.trackers.append(t)

    def update(self, frame) -> list[tuple[float, float, float, float]]:
        boxes = []
        for t in self.trackers:
            success, box = t.update(frame)
            if success:
                boxes.append(box)
        return boxes


# Let's visualize the MultiTracker!
mt = MultiTrackerWrapper()
mt.add_tracker(frames[0], (2, 2, 36, 36))
mt.add_tracker(frames[0], (50, 50, 30, 30))
multi_boxes = []
for f in frames[1:10]:
    multi_boxes = mt.update(f)
annotated_multi = frames[9].copy()
for b in multi_boxes:
    x, y, w, h = [int(v) for v in b]
    cv2.rectangle(annotated_multi, (x, y), (x + w, y + h), (0, 255, 255), 2)
show_grid([("Multi-Tracker Output (Frame 10)", annotated_multi)])

## Summary

You can initialize, visualize, and evaluate a single-object tracker and know when it must be re-detected.

- **Best Practices:** Expose tracker confidence/failure, retain a recovery path, and use stable IDs only within clearly documented assumptions.
- **Common Pitfalls:** Calling an initial detection a persistent identity, ignoring occlusion, and trusting tracker output after it drifts.